# 03 – SQL on the portfolio database

Core SQL on my own data: `SELECT`, `WHERE`, `ORDER BY`, `GROUP BY`, `HAVING` and `JOIN`, ending with data-quality checks. The database is built by notebook 02 (`data/processed/portfolio.db`).

## Setup

In [14]:
import sqlite3
from pathlib import Path

import pandas as pd

In [15]:
DB_PATH = Path("../data/processed/portfolio.db")
assert DB_PATH.exists(), "Run notebook 02 first"


def query(sql):
    """Run a SQL query on the portfolio database and return a DataFrame."""
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

In [16]:
query("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,backtest_returns
1,pie_returns
2,performance_full
3,real_returns
4,risk_correlation
5,risk_rolling
6,risk_real_rate
7,risk_var
8,risk_contribution
9,risk_stress


## 1. SELECT and LIMIT

`SELECT` chooses columns, `FROM` the table, `LIMIT` the number of rows. SQL keywords in capitals is a convention, not a rule.

In [17]:
query("""
SELECT ticker, name, pie, weight_in_portfolio
FROM holdings
LIMIT 5
""")

,ticker,name,pie,weight_in_portfolio
0,V,Visa,Kwaliteit,0.0189
1,MCO,Moody's,Kwaliteit,0.0189
2,LLY,Eli Lilly,Kwaliteit,0.0189
3,RMS,Hermès International,Kwaliteit,0.0189
4,REL,RELX,Kwaliteit,0.0162


## 2. WHERE: filtering rows

`WHERE` keeps rows that meet a condition. Combine conditions with `AND` / `OR`; text goes between single quotes.

In [18]:
query("""
SELECT ticker, name, weight_in_portfolio
FROM holdings
WHERE pie = 'Energie' AND weight_in_portfolio > 0.015
""")

,ticker,name,weight_in_portfolio
0,WENS,iShares MSCI World Energy Sector (Dist),0.0486
1,LNG,Cheniere Energy,0.0180
2,LIN,Linde,0.0162
3,CEG,Constellation Energy,0.0162


## 3. ORDER BY: sorting

`ORDER BY column DESC` sorts from large to small. Combined with `LIMIT` this gives a top-n.

In [19]:
query("""
SELECT ticker, pie, weight_in_portfolio
FROM holdings
ORDER BY weight_in_portfolio DESC
LIMIT 10
""")

,ticker,pie,weight_in_portfolio
0,GDX,Mijnbouw,0.0544
1,WENS,Energie,0.0486
2,EPIN,Multipolaire Wereldorde,0.0375
3,LOCK,Thematisch,0.0364
4,GRID,Thematisch,0.0351
5,WQTM,Thematisch,0.0286
6,AMEL,Multipolaire Wereldorde,0.0270
7,SILV,Mijnbouw,0.0255
8,CHN,Multipolaire Wereldorde,0.0225
9,IUSS,Multipolaire Wereldorde,0.0225


## 4. GROUP BY and HAVING: aggregating

`GROUP BY` makes one row per group; `SUM`, `COUNT`, `AVG`, `MIN`, `MAX` summarise each group. `WHERE` filters rows **before** grouping, `HAVING` filters groups **after** grouping. This is the SQL version of `groupby` in pandas.

In [20]:
query("""
SELECT pie, block, COUNT(*) AS n_positions, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY pie, block
ORDER BY block, weight DESC
""")

,pie,block,n_positions,weight
0,Kwaliteit,A,17,0.27
1,Thematisch,A,5,0.13
2,AI Boom,A,8,0.10
3,Energie,B,10,0.18
4,Mijnbouw,B,9,0.17
5,Multipolaire Wereldorde,B,6,0.15


In [21]:
query("""
SELECT country, SUM(weight_in_portfolio) AS weight
FROM holdings
GROUP BY country
HAVING SUM(weight_in_portfolio) > 0.05
ORDER BY weight DESC
""")

,country,weight
0,Ierland,0.3871
1,VS,0.2945
2,Canada,0.0916
3,Zwitserland,0.0594


## 5. JOIN: combining tables

`JOIN ... ON` links rows of two tables on a shared column, exactly like `merge` in pandas. Short table aliases (`h`, `s`) keep the query readable. `LEFT JOIN` keeps every row of the left table, even without a match.

In [22]:
query("""
SELECT h.ticker, h.pie, h.weight_in_portfolio, s.yahoo_symbol, s.series_type, s.proxy_symbol
FROM holdings AS h
JOIN symbols AS s ON h.ticker = s.ticker
WHERE s.series_type = 'alt_listing' OR s.proxy_symbol IS NOT NULL
ORDER BY h.weight_in_portfolio DESC
""")

,ticker,pie,weight_in_portfolio,yahoo_symbol,series_type,proxy_symbol
0,WENS,Energie,0.0486,WENS.L,listing,WNRG.L
1,EPIN,Multipolaire Wereldorde,0.0375,EPI.L,alt_listing,EPI
2,GRID,Thematisch,0.0351,GRDU.L,alt_listing,GRID
3,WQTM,Thematisch,0.0286,WQTM.L,alt_listing,NaN
4,AMEL,Multipolaire Wereldorde,0.0270,ALAT.PA,alt_listing,NaN
5,SILV,Mijnbouw,0.0255,SILV.MI,listing,SIL
6,CHN,Multipolaire Wereldorde,0.0225,ICHN.AS,alt_listing,NaN
7,IUSS,Multipolaire Wereldorde,0.0225,IKSA.L,alt_listing,NaN
8,LYXI,Multipolaire Wereldorde,0.0225,INDO.PA,alt_listing,NaN
9,VGEK,Multipolaire Wereldorde,0.0180,VAPU.L,alt_listing,NaN


Note `IS NOT NULL`: in SQL, `= NULL` never matches, just as `NaN == NaN` is `False` in pandas.

## 6. Putting it together on the returns table

How many trading days per ticker come from a proxy, and from which date does each ticker have data? `MIN`, `MAX` and a conditional count with `SUM(CASE WHEN ... THEN 1 ELSE 0 END)`.

In [23]:
query("""
SELECT ticker,
       MIN(date) AS first_date,
       COUNT(*) AS n_days,
       SUM(CASE WHEN source = 'proxy' THEN 1 ELSE 0 END) AS proxy_days
FROM returns_eur
GROUP BY ticker
HAVING proxy_days > 0 OR MIN(date) > '2019-02-01'
ORDER BY first_date
""")

,ticker,first_date,n_days,proxy_days
0,EPIN,2019-01-03 00:00:00,2026,1847
1,GRID,2019-01-03 00:00:00,2026,850
2,SILV,2019-01-03 00:00:00,2026,878
3,WENS,2019-01-03 00:00:00,2026,904
4,IUSS,2019-04-11 00:00:00,1956,0
5,CHN,2019-06-21 00:00:00,1906,0
6,VGEK,2019-09-25 00:00:00,1838,0
7,DAGB,2021-05-07 00:00:00,1421,0
8,COPX,2021-11-23 00:00:00,1279,0
9,CEG,2022-01-20 00:00:00,1237,0


The worst days of the **current** portfolio (a backtest with today's weights, so it contains hindsight). A JOIN with the weights, a weighted sum per date, sorted ascending. `SUM(h.weight_in_portfolio)` on the same rows shows the coverage of each day: positions without data do not count.

In [24]:
query("""
SELECT r.date,
       SUM(r.return_total * h.weight_in_portfolio) AS portfolio_return,
       SUM(h.weight_in_portfolio) AS coverage
FROM returns_eur AS r
JOIN holdings AS h ON r.ticker = h.ticker
GROUP BY r.date
ORDER BY portfolio_return ASC
LIMIT 5
""")

,date,portfolio_return,coverage
0,2020-03-12 00:00:00,-0.084360,0.9120
1,2020-03-09 00:00:00,-0.083811,0.9120
2,2020-03-16 00:00:00,-0.073022,0.9120
3,2020-03-18 00:00:00,-0.060061,0.9120
4,2025-04-04 00:00:00,-0.058012,0.9714


## Data checks in SQL

Every holding has a row in the symbol mapping. A `LEFT JOIN` keeps all holdings; rows without a match get `NULL` in the mapping columns. Expected result: empty.

In [25]:
query("""
SELECT h.ticker
FROM holdings AS h
LEFT JOIN symbols AS s ON h.ticker = s.ticker
WHERE s.ticker IS NULL
""")

,ticker


Trading days per ticker that come from a proxy (a fund on the same or a near-identical index, used before the held fund's data starts). EPIN relies on its US proxy until February 2026, so most of its backtest history is not the fund actually held.

In [26]:
query("""
SELECT r.ticker,
       COUNT(*) AS proxy_days,
       MIN(r.date) AS first_proxy_day,
       MAX(r.date) AS last_proxy_day
FROM returns_eur AS r
WHERE r.source = 'proxy'
GROUP BY r.ticker
ORDER BY proxy_days DESC
""")

,ticker,proxy_days,first_proxy_day,last_proxy_day
0,EPIN,1847,2019-01-03 00:00:00,2026-02-24 00:00:00
1,WENS,904,2019-01-03 00:00:00,2022-06-30 00:00:00
2,SILV,878,2019-01-03 00:00:00,2022-05-25 00:00:00
3,GRID,850,2019-01-03 00:00:00,2022-04-14 00:00:00
